# Exploratory: M2 SegFormer with the larger MiT-B2 encoder (1 seed, Florence only)

**Runtime: A100 GPU** (an L4 works but takes about twice as long). *Runtime > Run all*, then leave the tab open.
About **1.5-2 h** on an A100: 10-15 min data copy, then about 20 epochs of training. Rerunning resumes from the last
saved epoch.

**Why.** Our M2 used MiT-B1 (14.8 M parameters) while M1 has 25.5 M, so the CNN-vs-transformer comparison was not
size-matched. M2 with MiT-B2 has 25.9 M parameters, about the same as M1. Everything else is the M2 recipe: weighted CE,
encoder lr 6e-5, decoder lr 6e-4, 20 epochs, batch 8, seed 0, best epoch chosen on Florence.

**Rules.** Exploratory and post-hoc: one seed, Florence (validation) only, **no Nepal**. The final model does not
change. It is saved to `exploratory/checkpoints/` on Drive, away from the main runs.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os; NCPU = os.cpu_count(); NW = max(2, min(10, NCPU - 2)); print("CPUs", NCPU, "-> workers", NW)

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
CPUs 12 -> workers 10


In [2]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
OUT = f"{D}/exploratory/checkpoints"
RUN = "exploratory_m2_segformer_b2_ce_s0"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
os.makedirs(OUT, exist_ok=True)
!ls "{OUT}"

Mounted at /content/drive


In [3]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
1c67980 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #31 from aaarrvind/m3/executed-notebooks
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [4]:
# Training (Harvey, Matthew, Midwest) and Florence crops are spread over train, test and hold. Nepal (tier3) is not copied.
if not os.path.exists(f"{X}/hold"):
    !mkdir -p /content/xbd
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!ls /content/xbd

hold  rscc_flood_manifest.csv  test  train


In [5]:
# Size check: M2 with MiT-B2 vs M1 (expect about 25.9 M vs 25.5 M)
from floodsitrep.cv.models import build_model
n = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
b2 = build_model({"model": "siamese_segformer", "backbone": "nvidia/mit-b2", "pretrained": True})
print("encoder depths:", b2.encoder.config.depths, "| widths:", b2.encoder.config.hidden_sizes)
print(f"M2 MiT-B2 {n(b2):.2f} M parameters | M1 {n(build_model({'model': 'siamese_unet', 'pretrained': False})):.2f} M")
del b2

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 99.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b2
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


encoder depths: [3, 4, 6, 3] | widths: [64, 128, 320, 512]
M2 MiT-B2 25.88 M parameters | M1 25.50 M


## Training (about 1.5 h on an A100; the Florence score of every epoch is printed)

In [6]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{OUT}" --set run_name={RUN} backbone=nvidia/mit-b2 loss=ce lr=6e-4 encoder_lr_mult=0.1 num_workers={NW}

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

run exploratory_m2_segformer_b2_ce_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
Loading weights: 100% 364/364 [00:00<00:00, 15818.71it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b2
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  epoch 0 step 50/686 loss 1.2867 (running 1.5091)
  epoch 0 step 100/686 loss 0.9230 (running 1.3268)
  epoch 0 step 150/686 loss 0.3366 (running 1.1373)
  epoch 0 step 200/686 loss 1.1050 (running 0.9975)
  epoch 0 step 250/686 loss 0.6210 (running 0.9071)
  epoch 0 step 300/686 loss 0.9221 (running 0.8346)
  epoch 0 step 350/686 loss 0.2204 (running 0.7733)
  epoch 0 step 400/686 loss 0.4065 (running 0.7244)
  epoch 0 step 450/686 loss 0.2571 (running 0.6808)
  epoch 0 step 500/686 loss 0.2204 (running 0.650

## Compare with M2 (MiT-B1, 3 seeds) and M1 weighted CE (3 seeds), Florence
**How to read it.** The three B1 seeds scored 0.334 to 0.362. A B2 score inside that range says nothing either way.
Above it is only a hint (one seed), not a result. If B2's best epoch is again early (before about epoch 7) and the score
falls afterwards, the overfitting seen with B1 persists at the larger size.

In [7]:
import json, pandas as pd
def best(path):
    df = pd.DataFrame([json.loads(l) for l in open(path)])
    b = df.loc[df["xview2_pixel_score"].idxmax()]
    return df, {"best_epoch": int(b.epoch), "score": b.xview2_pixel_score, "loc": b.loc_f1, "minor": b.dmg_f1_minor,
                "major": b.dmg_f1_major, "destroyed": b.dmg_f1_destroyed, "last_score": df.xview2_pixel_score.iloc[-1]}
rows, curves = [], {}
df, r = best(f"{OUT}/{RUN}/metrics.jsonl"); rows.append({"model": "M2 MiT-B2 (exploratory)", "seed": 0, **r})
curves["M2 B2 s0"] = df.set_index("epoch").xview2_pixel_score
for s in range(3):
    df, r = best(f"{CK}/m2_siamese_segformer_ce_s{s}/metrics.jsonl"); rows.append({"model": "M2 MiT-B1", "seed": s, **r})
    curves[f"M2 B1 s{s}"] = df.set_index("epoch").xview2_pixel_score
    _, r = best(f"{CK}/m1_siamese_unet_ce_s{s}/metrics.jsonl"); rows.append({"model": "M1 weighted CE", "seed": s, **r})
T = pd.DataFrame(rows)
print(T.round(3).to_string(index=False)); print()
print(T.groupby("model", sort=False)[["score", "loc", "minor", "major", "destroyed"]].mean().round(3)); print()
print("Florence score per epoch:"); print(pd.DataFrame(curves).round(3).to_string())

                  model  seed  best_epoch  score   loc  minor  major  destroyed  last_score
M2 MiT-B2 (exploratory)     0          10  0.346 0.822  0.093  0.781      0.067       0.322
              M2 MiT-B1     0           2  0.362 0.774  0.166  0.599      0.080       0.289
         M1 weighted CE     0          12  0.363 0.782  0.100  0.449      0.124       0.356
              M2 MiT-B1     1           6  0.334 0.802  0.111  0.704      0.055       0.301
         M1 weighted CE     1          12  0.363 0.803  0.086  0.519      0.123       0.349
              M2 MiT-B1     2           5  0.338 0.806  0.171  0.758      0.048       0.269
         M1 weighted CE     2          18  0.356 0.803  0.100  0.532      0.090       0.353

                         score    loc  minor  major  destroyed
model                                                         
M2 MiT-B2 (exploratory)  0.346  0.822  0.093  0.781      0.067
M2 MiT-B1                0.345  0.794  0.150  0.687      0.061
M1 weighted

Paste the size check, the last training lines and the comparison output in the chat. Then stop billing:

In [8]:
from google.colab import runtime
runtime.unassign()